In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import torch
import re
from torch import nn
import spacy 
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
nlp = spacy.load('en_core_web_lg')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def preprocess(text):
    
    # remove URLs
    text = re.sub('http\S*', ' ', text)
    
    # remove non-alphabetic
    text = re.sub("[^a-zA-Z]", " ", text)
    
    # make lowercase
    text = text.lower()

    # remove one character word
    text = re.sub("\s+[a-zA-Z]\s+", ' ', text)
    text = re.sub("^[a-zA-Z]\s+", '', text)
    
    # replace double space to one space
    text = re.sub("\s+", ' ', text)
    
    # tokenize, lemmatize, remove stop words
    doc = nlp(text)
    text = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(text)

preprocess("She the soul of my soul")

In [ ]:
dataset = pandas.read_csv(r"/kaggle/input/nlp-getting-started/train.csv")[['text', 'target']]
dataset

In [ ]:
dataset['clean_text'] = dataset['text'].apply(preprocess)
dataset

In [ ]:
max([len(sent.split()) for sent in dataset['clean_text']])

In [ ]:
batch_size = 64
hidden_size = 32
learning_rate = 0.0005
num_layers = 5
n_epochs = 30

In [ ]:
def word2vec(sent):
    seq2d = []
    doc = nlp(sent)
    for token in doc:
        seq2d.append(token.vector)
    while len(seq2d) < 21:
        seq2d.append([0 for i in range(300)])
    return np.array(seq2d)

word2vec(dataset['clean_text'][0]).shape

In [ ]:
X = []
for sent in dataset['clean_text']:
    X.append(word2vec(sent))
X = np.array(X)
y = dataset['target'].values
X.shape

In [ ]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X, y, test_size = 0.3, random_state = 1234)
X_train_np.shape, X_test_np.shape, y_train_np.shape, y_test_np.shape

In [ ]:
X_train = torch.from_numpy(X_train_np.astype(np.float32))
X_test = torch.from_numpy(X_test_np.astype(np.float32))
y_train = torch.from_numpy(y_train_np.astype(np.int64))
y_test = y_test_np.astype(np.int64)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset = train_ds, batch_size = 100, shuffle = True)
print(next(iter(train_loader))[0].shape, next(iter(train_loader))[0].shape)

In [ ]:
class LSTM(nn.Module):
    def __init__(self, sequence_length, hidden_size, num_layers):
        super(LSTM, self).__init__()
        self.sequence_length = sequence_length
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(sequence_length, hidden_size = self.hidden_size, num_layers = self.num_layers, batch_first = True, bidirectional = True)
        self.linear = nn.Linear(self.hidden_size * 2, 2)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.linear(out[:, -1, :])
        return out

In [ ]:
# model
model = LSTM(X.shape[2], hidden_size, num_layers).to(device)

# loss
critirion = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:
for epoch in range(n_epochs):
    loss_sum = 0
    for (samples, labels) in train_loader:
        
        samples = samples.to(device)
        labels = labels.to(device)
        
        # forward
        predictions = model(samples)
        loss = critirion(predictions, labels)
        
        # loss
        loss.backward()
        
        # update
        optimizer.step()
        optimizer.zero_grad()
        
        loss_sum += loss
    print(f"epoch = {epoch} | loss = {loss_sum:.4f}")

In [ ]:
def predict(x):
    x = x.to(device)
    y_pred = model(x.squeeze(1)).max(dim = 1)[1]
    return y_pred.cpu().detach().numpy()

In [ ]:
y_pred = predict(X_test)
(y_pred == y_test).sum() / len(y_test)

In [ ]:
test_ds = pandas.read_csv(f"/kaggle/input/nlp-getting-started/test.csv")
test_ds

In [ ]:
test_ds['clean_text'] = test_ds['text'].apply(preprocess)

In [ ]:
test_ds

In [ ]:
X_submit = []
for sent in test_ds['clean_text']:
    X_submit.append(word2vec(sent))
X_submit = torch.tensor(np.array(X_submit), dtype = torch.float32)
X_submit.shape

In [ ]:
predictions = predict(X_submit)
predictions.shape, predictions

In [ ]:
submission = pandas.DataFrame({
    "id":test_ds['id'],
    "target":predictions
})
submission

In [ ]:
submission.to_csv('submission.csv', index = False)